In [ ]:
"""Cell 1 — подготовка датасета для сравнения baseline RAG vs agent.

Переиспользуем 10 single-hop вопросов из golden-датасета прошлой недели
(`rag_eval.ipynb`, Ячейка 2) — вставлены инлайн, без чтения из файла — и
добавляем 5 multi-hop, таких, которые требуют последовательного вызова
двух tool'ов.
"""
import json
import time
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# 10 single-hop вопросов из golden-датасета прошлой недели (rag_eval.ipynb,
# Ячейка 2) — тот же список question/ground_truth, вставленный инлайн
golden = [
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
    },
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
    },
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
    },
]

# 5 multi-hop сценариев: lookup в документации + вычисление в python_repl
MULTI_HOP = [
    {"question": "What is the default alpha in Ridge regression? Compute alpha * 10 with python_repl"},
    {"question": "Find the default max_depth for DecisionTreeClassifier in the docs, then compute 2**10 in python_repl"},
    {"question": "What is L2 penalty formula for Ridge? Calculate it for alpha=0.5 and w=[1,2,3]"},
    {"question": "What is class_weight in LogisticRegression? Compute 1/3 with python_repl as fraction"},
    {"question": "What is the F1 score formula? Compute F1 for precision=0.8 and recall=0.6 with python_repl"},
]
ALL = golden + MULTI_HOP
print(f"Готово: {len(golden)} single-hop + {len(MULTI_HOP)} multi-hop = {len(ALL)} вопросов")

In [ ]:
"""Cell 2 — baseline: ходим в `/chat` и меряем качество чистого RAG.

На multi-hop вопросах baseline проваливается — это ожидаемо. Цифры идут
в `baseline_metrics.json` для последующего сравнения с agent в Cell 4.
"""
from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy

from app.llm import get_llm

BASE_URL = "http://localhost:8000"
baseline_results = []

# Дёргаем /chat синхронно на каждом вопросе и копим контекст для RAGAS
for i, item in enumerate(ALL):
    r = httpx.post(f"{BASE_URL}/chat", json={"question": item["question"]}, timeout=60)
    data = r.json()
    baseline_results.append({
        "user_input": item["question"],
        "response": data["answer"],
        "retrieved_contexts": [s["full_context"] for s in data["sources"]],
        "reference": item.get("ground_truth", ""),
    })
    print(f"[{i + 1}/{len(ALL)}] baseline ok")

# Считаем RAGAS-метрики через LLM-judge — обёртки требуются для совместимости
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)
baseline_scores = evaluate(
    dataset=Dataset.from_list(baseline_results),
    metrics=[Faithfulness(llm=llm), ResponseRelevancy(llm=llm, embeddings=emb)],
)
baseline_df = baseline_scores.to_pandas()
Path("notebooks/baseline_metrics.json").write_text(
    json.dumps({
        "faithfulness": float(baseline_df["faithfulness"].mean()),
        "answer_relevancy": float(baseline_df["answer_relevancy"].mean()),
        "n_questions": len(ALL),
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(baseline_df[["faithfulness", "answer_relevancy"]].mean())

In [ ]:
"""Cell 3 — agent: ходим в `/agent` на тех же 15 вопросах.

В отличие от baseline собираем не только Faithfulness/Relevancy, но и
agent-специфичные метрики — какие tool'ы выбрал и сколько итераций
потребовалось.
"""
agent_results = []
agent_meta = []

for i, item in enumerate(ALL):
    r = httpx.post(
        f"{BASE_URL}/agent",
        json={"question": item["question"], "thread_id": f"eval-{i}"},
        timeout=120,
    )
    data = r.json()
    agent_results.append({
        "user_input": item["question"],
        "response": data["answer"],
        "retrieved_contexts": [s["full_context"] for s in data.get("sources", [])] or [data["answer"]],
        "reference": item.get("ground_truth", ""),
    })
    # Список разных tool'ов в trace — для multi-hop ждём ≥ 2
    tools_used = {step["tool"] for step in data["trace"] if step.get("tool")}
    agent_meta.append({
        "tools_used": sorted(tools_used),
        "n_tools": len(tools_used),
        "iterations": data["iterations"],
        "is_multi_hop": i >= len(golden),
    })
    print(f"[{i + 1}/{len(ALL)}] agent ok · tools={sorted(tools_used)} · iter={data['iterations']}")
    time.sleep(2)  # бережём rate-limit DuckDuckGo и LLM-провайдера

agent_scores = evaluate(
    dataset=Dataset.from_list(agent_results),
    metrics=[Faithfulness(llm=llm), ResponseRelevancy(llm=llm, embeddings=emb)],
)
agent_df = agent_scores.to_pandas()

# tool_choice_accuracy: на multi-hop правильным считаем вызов ≥ 2 tool'ов
mh_meta = [m for m in agent_meta if m["is_multi_hop"]]
tool_choice_accuracy = sum(1 for m in mh_meta if m["n_tools"] >= 2) / max(1, len(mh_meta))

Path("notebooks/agent_metrics.json").write_text(
    json.dumps({
        "faithfulness": float(agent_df["faithfulness"].mean()),
        "answer_relevancy": float(agent_df["answer_relevancy"].mean()),
        "tool_choice_accuracy_multi_hop": tool_choice_accuracy,
        "avg_iterations": sum(m["iterations"] for m in agent_meta) / len(agent_meta),
        "n_questions": len(ALL),
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(agent_df[["faithfulness", "answer_relevancy"]].mean())
print(f"tool_choice_accuracy (multi-hop): {tool_choice_accuracy:.2f}")

In [ ]:
"""Cell 4 — сравниваем baseline и agent.

Сводим всё в pandas-таблицу, печатаем в ноутбуке и заодно генерируем
markdown для README. На single-hop ждём близкие числа; на multi-hop —
ощутимый отрыв agent в сторону больших баллов.
"""
baseline_metrics = json.loads(Path("notebooks/baseline_metrics.json").read_text())
agent_metrics = json.loads(Path("notebooks/agent_metrics.json").read_text())

summary = pd.DataFrame([
    {
        "Metric": "Faithfulness (all 15)",
        "Baseline RAG": round(baseline_metrics["faithfulness"], 3),
        "Agent": round(agent_metrics["faithfulness"], 3),
    },
    {
        "Metric": "AnswerRelevancy (all 15)",
        "Baseline RAG": round(baseline_metrics["answer_relevancy"], 3),
        "Agent": round(agent_metrics["answer_relevancy"], 3),
    },
    {
        "Metric": "Tool-choice accuracy (multi-hop)",
        "Baseline RAG": "—",
        "Agent": f"{agent_metrics['tool_choice_accuracy_multi_hop']:.0%}",
    },
    {
        "Metric": "Avg iterations",
        "Baseline RAG": 1,
        "Agent": round(agent_metrics["avg_iterations"], 1),
    },
])

# Pretty-print + markdown для README
print(summary.to_string(index=False))
Path("notebooks/comparison.md").write_text(
    "# Agent vs Baseline RAG\n\n" + summary.to_markdown(index=False) + "\n",
    encoding="utf-8",
)